
# 😴 Detecting Sleep Apnea & Estimating the AHI
### A self-contained Colab on the *ApneaEye* idea: scoring breathing events from two signals
**ACM Summer School on AI for Health — Hands-on Notebook**
*Prepared by Nipun Batra's group*

---

**Sleep apnea** is a disorder of repeated *pauses* or *shallow breaths* during sleep. It is staggeringly common — roughly **1 billion people** are affected worldwide, and **more than 80% remain undiagnosed**. Left untreated it is linked to **cardiovascular disease, metabolic dysfunction, impaired brain function, and depression**.

The clinical gold standard is **Polysomnography (PSG)** — an overnight sleep study that wires the patient up with many on-body sensors (EEG, ECG, a nasal cannula, chest and abdomen belts, a pulse oximeter, and more). PSG is accurate but expensive, intrusive, and scarce — a big reason so many cases never get diagnosed.

Crucially, the **AASM** (American Academy of Sleep Medicine) scoring guidelines say you fundamentally need just **two signals** to *score a breathing event*:
1. **Nasal AIRFLOW** — how much air is actually moving through the nose/mouth.
2. **THORAX–ABDOMEN movement** — whether the chest and belly are still *trying* to breathe.

The **ApneaEye** project shows you can recover *both* of these from a single **low-resolution thermal camera** pointed at a sleeping person, with **no sensors on the body**:
- *Nasal airflow* from the **temperature** fluctuation around the nostrils (warm exhale, cool inhale).
- *Thorax–abdomen movement* from the **optical flow** of the chest/belly rising and falling.

This notebook teaches the **core signal-processing technique**: given those two breathing signals, how do we automatically (a) detect apnea/hypopnea events, (b) tell *central* from *obstructive* apnea, and (c) compute the **AHI** that grades severity? Everything below is **fully synthetic** — it runs end-to-end with **no upload and no data**.


## The three breathing events (the rules we will implement)

The genius of using *two* signals is that the **combination** of "airflow" and "effort" tells you *which kind* of event is happening:

| Event | Nasal airflow | Thorax–abdomen movement | What is going wrong |
|---|---|---|---|
| **Central apnea** | drops **≥ 80%** | **STOPS** | the brain (CNS) momentarily stops driving the breathing muscles — no effort, no air |
| **Obstructive apnea** | drops **≥ 80%** | **PERSISTS** | the airway is physically blocked — the body keeps *trying* (effort continues) but no air gets through |
| **Hypopnea** | drops **~ ≥ 30%** (partial) | drops **~ 30%** (partial) | breathing becomes shallow but not absent |

**An event only counts if it lasts at least 10 seconds.** A momentary dip does not count.

### From events to severity: the AHI

The **Apnea–Hypopnea Index** summarises a whole night in one number:

> **AHI = (number of apnea events + number of hypopnea events) / sleep duration in hours**

| AHI | Severity |
|---|---|
| < 5 | Normal |
| 5 – 15 | Mild |
| 15 – 30 | Moderate |
| > 30 | Severe |

> ⚠️ **This is an educational demo, not a medical device.** Nothing here is validated for diagnosis. Do not use it to make any health decision.


## What this notebook does

We never touch a real camera. Instead we **synthesise** the two breathing signals for a compressed "night", inject a *known* list of events, and then run the detector — so we can check every detected event and every AHI against the ground truth.

```
thermal video  (we simulate the two signals it would yield)
   → nasal airflow      (oscillates at the breathing rate; temperature proxy)
   → thorax–abdomen     (oscillates at the breathing rate; optical-flow proxy)

each signal  → breathing AMPLITUDE envelope  → rolling MEDIAN baseline ("normal")
            → fraction of normal amplitude remaining  (1.0 = fine, 0.0 = stopped)
            → slide a ≥ 10 s window + apply the AASM rules above
            → events { central apnea | obstructive apnea | hypopnea }
            → count events / sleep hours = AHI → severity
```


## 1. Setup
Everything below is pre-installed on Colab. The `pip install` is a no-op there but keeps the notebook portable. We use only **numpy**, **scipy**, and **matplotlib** — no external data, no upload.

In [ ]:
# Colab already has these; the install is just for portability.
try:
    import scipy  # noqa
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "scipy", "matplotlib"], check=True)

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import hilbert
from numpy.lib.stride_tricks import sliding_window_view

np.set_printoptions(suppress=True)
plt.rcParams.update({"figure.figsize": (11, 3.4), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print("Running in Colab:", IN_COLAB)


## 2. Synthesise an overnight recording (with a known ground truth)

We build two signals sampled at **fs = 4 Hz** over a compressed night (default **60 minutes**, configurable). Each is an **amplitude-modulated breathing oscillation** at about **0.25 Hz (~15 breaths/min)**:
- `airflow` — the nasal-airflow proxy.
- `thorax` — the thorax–abdomen movement proxy.

We then inject a seeded list of events. Inside each event we scale the breathing *amplitude* down according to the rules:
- **central apnea** → both signals go to ~0,
- **obstructive apnea** → airflow goes to ~0 but thorax *persists*,
- **hypopnea** → both drop partially (about 30–50%).

We keep the exact **ground-truth list** `{start, end, type}`. We also sprinkle in a couple of unscored **"decoy" dips** (brief breathing irregularities such as a sigh) that a real detector must learn to cope with — they create the occasional false positive.

In [ ]:
FS = 4.0            # Hz, sampling rate of the two camera-derived signals
BREATH_HZ = 0.25    # ~15 breaths per minute
NIGHT_MIN = 60.0    # compressed "night" length in minutes (configurable)

EVENT_TYPES = ("central", "obstructive", "hypopnea")


def _amp_factor(rng, etype):
    """Amplitude scaling (air, thorax) DURING an event, as a fraction of
    the person's normal breathing amplitude."""
    if etype == "central":        # airflow gone AND thorax stops
        return rng.uniform(0.03, 0.10), rng.uniform(0.03, 0.10)
    if etype == "obstructive":    # airflow gone BUT thorax keeps trying
        return rng.uniform(0.03, 0.10), rng.uniform(0.90, 1.05)
    return rng.uniform(0.45, 0.60), rng.uniform(0.55, 0.70)   # hypopnea: partial


def _place_events(specs, night_s, rng, min_gap=18.0, edge=75.0):
    """Place each (type, duration) event at a random, non-overlapping time."""
    placed = []
    for etype, dur in specs:
        for _ in range(3000):
            start = rng.uniform(edge, night_s - edge - dur)
            end = start + dur
            clash = any(not (end + min_gap < s or start - min_gap > e)
                        for s, e, _ty in placed)
            if not clash:
                placed.append((start, end, etype))
                break
    placed.sort(key=lambda r: r[0])
    return [{"start": s, "end": e, "type": t} for s, e, t in placed]


def make_overnight(n_central=5, n_obstructive=7, n_hypopnea=10, n_decoy=0,
                   night_min=NIGHT_MIN, fs=FS, noise=0.04, seed=0):
    """Synthesize two breathing signals for one compressed night.

    Returns (t, airflow, thorax, gt_events) where gt_events is the GROUND-TRUTH
    list of dicts {start, end, type}. Decoy dips are NOT in the ground truth."""
    rng = np.random.default_rng(seed)
    night_s = night_min * 60.0
    t = np.arange(0, night_s, 1.0 / fs)
    n = t.size

    # Slowly wandering breathing frequency -> realistic but near BREATH_HZ.
    fwander = BREATH_HZ * (1.0 + 0.10 * np.sin(2 * np.pi * t / 240.0))
    phase = 2 * np.pi * np.cumsum(fwander) / fs

    # Normal breathing amplitude drifts slowly around 1.0 for each signal.
    amp_air = 1.0 + 0.12 * np.sin(2 * np.pi * t / 300.0 + 0.3)
    amp_tho = 1.0 + 0.12 * np.sin(2 * np.pi * t / 280.0 + 1.1)

    # Build the list of real events plus unscored decoys, then place them.
    specs = []
    for etype, k in (("central", n_central), ("obstructive", n_obstructive),
                     ("hypopnea", n_hypopnea)):
        for _ in range(k):
            specs.append((etype, rng.uniform(14.0, 30.0)))
    rng.shuffle(specs)
    specs += [("decoy", rng.uniform(12.5, 18.0)) for _ in range(n_decoy)]
    all_events = _place_events(specs, night_s, rng)

    mask_air = np.ones(n)     # 1.0 = normal amplitude, scaled down during events
    mask_tho = np.ones(n)
    gt_events = []
    for ev in all_events:
        i0, i1 = int(ev["start"] * fs), int(ev["end"] * fs)
        seg_len = i1 - i0
        if ev["type"] == "decoy":
            fa, ft = rng.uniform(0.45, 0.60), rng.uniform(0.55, 0.72)
        else:
            fa, ft = _amp_factor(rng, ev["type"])
            gt_events.append(ev)
        ramp = min(int(1.5 * fs), seg_len // 2)
        shape = np.ones(seg_len)            # 1 in the middle, ramps to 0 at edges
        if ramp > 0:
            shape[:ramp] = np.linspace(0.0, 1.0, ramp)
            shape[seg_len - ramp:] = np.linspace(1.0, 0.0, ramp)
        mask_air[i0:i1] = 1.0 - shape * (1.0 - fa)
        mask_tho[i0:i1] = 1.0 - shape * (1.0 - ft)

    airflow = mask_air * amp_air * np.sin(phase) + noise * rng.standard_normal(n)
    thorax = mask_tho * amp_tho * np.sin(phase - 0.4) + noise * rng.standard_normal(n)
    gt_events.sort(key=lambda e: e["start"])
    return t, airflow, thorax, gt_events


t, airflow, thorax, GT = make_overnight(n_central=5, n_obstructive=7,
                                         n_hypopnea=10, n_decoy=2, seed=7)
hours = (t[-1] + 1.0 / FS) / 3600.0
print(f"Night length       : {hours*60:.0f} min  ({len(t)} samples @ {FS:.0f} Hz)")
print(f"Ground-truth events: {len(GT)}")
for et in EVENT_TYPES:
    print(f"   {et:12s}: {sum(e['type'] == et for e in GT)}")


## 3. Breathing amplitude envelope + a "normal" baseline

A breathing signal is a fast oscillation whose *amplitude* tells us how strongly the person is breathing. We extract that amplitude two ways combined:
1. The **Hilbert envelope** `|analytic signal|` gives the instantaneous amplitude.
2. A short **moving average** (~one breath) removes the within-breath ripple.

To decide what counts as a *drop*, we need a reference for "normal". We use a **rolling median** of the envelope over a ~2-minute window. The median ignores the short events (they are a minority of any 2-minute window), so it tracks the person's *typical* amplitude. Dividing the envelope by this baseline gives the **fraction of normal breathing remaining** — exactly the quantity the AASM rules are phrased in (`≥ 80% drop`, `≥ 30% drop`).

In [ ]:
def breathing_envelope(x, fs, smooth_s=5.0):
    """Per-sample breathing AMPLITUDE: |Hilbert| smoothed over ~one breath."""
    amp = np.abs(hilbert(x))
    w = max(1, int(smooth_s * fs))
    return np.convolve(amp, np.ones(w) / w, mode="same")


def rolling_baseline(env, fs, win_s=120.0):
    """'Normal' amplitude = rolling MEDIAN over a ~2 min window. The median
    ignores short dips, so it tracks typical breathing, not the events."""
    win = int(win_s * fs)
    if win % 2 == 0:
        win += 1
    half = win // 2
    padded = np.pad(env, half, mode="edge")
    return np.median(sliding_window_view(padded, win), axis=1)


env_air = breathing_envelope(airflow, FS)
env_tho = breathing_envelope(thorax, FS)
base_air = rolling_baseline(env_air, FS)
base_tho = rolling_baseline(env_tho, FS)

# Fraction of normal amplitude remaining (1.0 = normal, 0.0 = no breathing).
ratio_air = env_air / np.maximum(base_air, 1e-6)
ratio_tho = env_tho / np.maximum(base_tho, 1e-6)

print("airflow ratio  -> median (normal breathing):", round(float(np.median(ratio_air)), 2))
print("airflow ratio  -> minimum (deepest event)  :", round(float(np.min(ratio_air)), 2))
print("thorax  ratio  -> median (normal breathing):", round(float(np.median(ratio_tho)), 2))


## 4. The event detector (the AASM rules in code)

Now we slide along the night and apply the rules literally:

1. Find every stretch where **airflow drops by ≥ 30%** (`ratio_air < 0.70`) for **at least 10 seconds** — a candidate event.
2. Within that stretch, look at how far airflow fell:
   - if airflow fell **≥ 80%** (`ratio_air < 0.20`) → it is an **APNEA**.
     Now split by *effort*: if thorax also collapsed (`ratio_tho < 0.20`) it is **CENTRAL**; if thorax kept moving it is **OBSTRUCTIVE**.
   - otherwise it is a **partial** drop → a **HYPOPNEA**, but only if the **thorax is also reduced** (`ratio_tho < 0.75`); a drop in airflow *alone* with full effort is treated as noise/a sigh and ignored.

Notice how *one extra signal* (effort) is what lets us separate central from obstructive — the whole reason ApneaEye insists on capturing both.

In [ ]:
def find_runs(mask, fs, min_dur_s=10.0):
    """Return [i0, i1) index runs where boolean mask stays True for at least
    min_dur_s seconds (the >= 10 s rule)."""
    runs = []
    n = len(mask)
    min_len = int(min_dur_s * fs)
    i = 0
    while i < n:
        if mask[i]:
            j = i
            while j < n and mask[j]:
                j += 1
            if j - i >= min_len:
                runs.append((i, j))
            i = j
        else:
            i += 1
    return runs


def detect_events(ratio_air, ratio_tho, fs, t,
                  apnea_thr=0.20, hypopnea_hi=0.70, thorax_drop=0.75,
                  min_dur_s=10.0):
    """Apply the AASM-style rules to the two amplitude ratios and return a
    list of detected events {start, end, type}."""
    reduced = ratio_air < hypopnea_hi          # airflow down >= 30%
    events = []
    for i0, i1 in find_runs(reduced, fs, min_dur_s):
        air_min = float(np.min(ratio_air[i0:i1]))
        tho_min = float(np.min(ratio_tho[i0:i1]))
        tho_med = float(np.median(ratio_tho[i0:i1]))
        if air_min < apnea_thr:                # airflow down >= 80% -> apnea
            etype = "central" if tho_min < apnea_thr else "obstructive"
        elif tho_med < thorax_drop:            # partial drop + effort reduced
            etype = "hypopnea"
        else:
            continue                           # airflow-only wobble -> ignore
        events.append({"start": float(t[i0]),
                       "end": float(t[min(i1, len(t) - 1)]),
                       "type": etype})
    return events


DET = detect_events(ratio_air, ratio_tho, FS, t)
print(f"Detected events: {len(DET)}   (ground truth: {len(GT)})")
for et in EVENT_TYPES:
    print(f"   {et:12s}: detected {sum(e['type'] == et for e in DET):2d}   "
          f"true {sum(e['type'] == et for e in GT):2d}")


## 5. How good is the detector? Precision, Sensitivity, Specificity

We compare the detector against the ground truth **sample by sample** (each sample is 0.25 s, since fs = 4 Hz). Every sample is either inside an event or not, for both the truth and the detector, giving the four counts:
- **TP** (true positive): sample is an event, and we flagged it.
- **FP** (false positive): sample is *not* an event, but we flagged it.
- **FN** (false negative): sample *is* an event, but we missed it.
- **TN** (true negative): sample is not an event, and we correctly left it alone.

The three metrics the ApneaEye talk reports are then:

> **Precision = TP / (TP + FP)** — of everything we *flagged*, how much was real (few false alarms).
>
> **Sensitivity (Recall) = TP / (TP + FN)** — of all the *real* events, how many we *caught* (few misses).
>
> **Specificity = TN / (TN + FP)** — of all the truly *normal* breathing, how much we correctly left unflagged.

In [ ]:
def event_mask(events, t, fs):
    """Boolean array (length of t): True wherever any event is active."""
    m = np.zeros(len(t), dtype=bool)
    for e in events:
        m[int(e["start"] * fs):int(e["end"] * fs)] = True
    return m


gt_mask = event_mask(GT, t, FS)
det_mask = event_mask(DET, t, FS)

TP = int(np.sum(gt_mask & det_mask))
FP = int(np.sum(~gt_mask & det_mask))
FN = int(np.sum(gt_mask & ~det_mask))
TN = int(np.sum(~gt_mask & ~det_mask))

precision = TP / (TP + FP) if (TP + FP) else 0.0
sensitivity = TP / (TP + FN) if (TP + FN) else 0.0
specificity = TN / (TN + FP) if (TN + FP) else 0.0

print("Per-sample confusion matrix (1 sample = 0.25 s):")
print(f"   TP = {TP:6d}    FP = {FP:6d}")
print(f"   FN = {FN:6d}    TN = {TN:6d}")
print("")
print(f"   Precision    (TP/(TP+FP)) : {precision*100:5.1f}%")
print(f"   Sensitivity  (TP/(TP+FN)) : {sensitivity*100:5.1f}%")
print(f"   Specificity  (TN/(TN+FP)) : {specificity*100:5.1f}%")


# ---- Event-level view (this is what feeds the AHI count) -------------------
def event_level_match(det, gt):
    """Greedy temporal-overlap match. Returns (tp, fp, fn, type_correct)."""
    used = set()
    tp = type_correct = 0
    for d in det:
        best, best_ov = -1, 0.0
        for k, g in enumerate(gt):
            if k in used:
                continue
            ov = max(0.0, min(d["end"], g["end"]) - max(d["start"], g["start"]))
            if ov > best_ov:
                best, best_ov = k, ov
        if best >= 0:
            used.add(best)
            tp += 1
            type_correct += int(d["type"] == gt[best]["type"])
    return tp, len(det) - tp, len(gt) - tp, type_correct


etp, efp, efn, tcorr = event_level_match(DET, GT)
print("")
print("Event-level matching (by temporal overlap):")
print(f"   matched (TP) = {etp}   false alarms (FP) = {efp}   missed (FN) = {efn}")
print(f"   correct apnea-type (central/obstructive/hypopnea): {tcorr}/{etp}")


## 6. AHI and severity

Now the payoff: collapse the whole night into the **AHI** and a severity grade, and compare the estimate to the truth.

In [ ]:
def classify_ahi(ahi):
    if ahi < 5:
        return "Normal"
    if ahi < 15:
        return "Mild"
    if ahi < 30:
        return "Moderate"
    return "Severe"


ahi_gt = len(GT) / hours
ahi_est = len(DET) / hours
print(f"Sleep duration   : {hours:.2f} h")
print("")
print(f"Ground-truth AHI : {ahi_gt:5.1f}  ->  {classify_ahi(ahi_gt)}")
print(f"Estimated AHI    : {ahi_est:5.1f}  ->  {classify_ahi(ahi_est)}")
print("")
if ahi_est >= ahi_gt:
    print("The estimate is >= the truth: errors lean toward OVER-estimation,")
    print("which is the safe direction (it avoids missing a sick patient).")
else:
    print("The estimate came in below the truth this time.")


## 7. A cohort table: estimated vs true severity across "participants"

The ApneaEye paper reports that severity was classified **correctly in about 89%** of participants, and — importantly — the few mistakes were almost always to a **higher** severity (a conservative error that avoids *under*-diagnosis).

Let us reproduce that flavour. We simulate a handful of participants with different true AHIs spanning all four severity grades, run the **exact same pipeline** on each, and tabulate true-vs-estimated severity. Watch the participant who sits just under a severity boundary get nudged up.

In [ ]:
def run_pipeline(airflow, thorax, t, fs):
    """Full envelope -> baseline -> detect chain used on the main night."""
    ea = breathing_envelope(airflow, fs)
    et = breathing_envelope(thorax, fs)
    ra = ea / np.maximum(rolling_baseline(ea, fs), 1e-6)
    rt = et / np.maximum(rolling_baseline(et, fs), 1e-6)
    return detect_events(ra, rt, fs, t)


SEV = {"Normal": 0, "Mild": 1, "Moderate": 2, "Severe": 3}

# (id, n_central, n_obstructive, n_hypopnea, n_decoy, seed)
cohort = [
    ("P01", 0, 1, 1, 0, 101),
    ("P02", 1, 1, 2, 0, 102),
    ("P03", 2, 2, 4, 0, 103),
    ("P04", 2, 4, 6, 0, 104),
    ("P05", 2, 4, 8, 2, 105),   # true AHI 14 (Mild) -> nudged up by 2 decoys
    ("P06", 3, 6, 10, 0, 106),
    ("P07", 5, 8, 13, 0, 107),
    ("P08", 6, 10, 18, 0, 108),
    ("P09", 8, 14, 22, 1, 109),
]

print(f"{'ID':4s} {'true AHI':>8s} {'true sev':>9s} {'est AHI':>8s} "
      f"{'est sev':>9s}   verdict")
print("-" * 56)
n_ok = n_wrong = n_higher = 0
for pid, c, o, h, d, sd in cohort:
    tt, af, th, gt = make_overnight(c, o, h, d, seed=sd)
    det = run_pipeline(af, th, tt, FS)
    hrs = (tt[-1] + 1.0 / FS) / 3600.0
    at, ae = len(gt) / hrs, len(det) / hrs
    st, se = classify_ahi(at), classify_ahi(ae)
    if st == se:
        verdict = "ok"
        n_ok += 1
    else:
        higher = SEV[se] > SEV[st]
        verdict = "OVER (conservative)" if higher else "UNDER (unsafe)"
        n_wrong += 1
        n_higher += int(higher)
    print(f"{pid:4s} {at:8.1f} {st:>9s} {ae:8.1f} {se:>9s}   {verdict}")

acc = 100.0 * n_ok / len(cohort)
print("-" * 56)
print(f"Severity correct in {n_ok}/{len(cohort)} = {acc:.0f}% of participants")
print(f"Of the {n_wrong} error(s), {n_higher} were to a HIGHER severity "
      f"(conservative, avoids under-diagnosis).")


## 8. Look at it: signals, ground truth, and what the detector found

The top two panels show the airflow and thorax signals over a few-minute window, with the **ground-truth events shaded by type**. The key visual: during an **obstructive** apnea the *airflow* collapses but the *thorax* keeps oscillating — that persistence is what flags it as obstructive rather than central.

The bottom panel shows the **airflow amplitude ÷ baseline** with the two decision thresholds, and the detector's events hatched on top.

In [ ]:
import matplotlib.patches as mpatches

COL = {"central": "#d62728", "obstructive": "#ff7f0e", "hypopnea": "#9467bd"}


def in_window(events, w0, w1):
    return [e for e in events if e["end"] > w0 and e["start"] < w1]


# A window containing the first few ground-truth events.
w0 = max(0.0, GT[0]["start"] - 25.0)
w1 = GT[min(3, len(GT) - 1)]["end"] + 25.0
ia, ib = int(w0 * FS), int(w1 * FS)
tw = t[ia:ib]

fig, ax = plt.subplots(3, 1, figsize=(12, 8.5), sharex=True)

ax[0].plot(tw, airflow[ia:ib], lw=0.8, color="#1f77b4")
ax[0].set(ylabel="airflow (a.u.)",
          title="Nasal airflow   (shaded = ground-truth events)")
for e in in_window(GT, w0, w1):
    ax[0].axvspan(e["start"], e["end"], color=COL[e["type"]], alpha=0.25, lw=0)
ax[0].legend(handles=[mpatches.Patch(color=COL[k], alpha=0.4, label=k)
                      for k in EVENT_TYPES], loc="upper right", fontsize=8)

ax[1].plot(tw, thorax[ia:ib], lw=0.8, color="#2ca02c")
ax[1].set(ylabel="thorax (a.u.)",
          title="Thorax-abdomen movement   (persists during OBSTRUCTIVE apnea)")
for e in in_window(GT, w0, w1):
    ax[1].axvspan(e["start"], e["end"], color=COL[e["type"]], alpha=0.25, lw=0)

ax[2].plot(tw, ratio_air[ia:ib], lw=1.1, color="#333333",
           label="airflow amplitude / baseline")
ax[2].axhline(0.20, color="r", ls="--", lw=1.2, label="apnea threshold (0.20)")
ax[2].axhline(0.70, color="orange", ls="--", lw=1.2,
              label="hypopnea threshold (0.70)")
for e in in_window(DET, w0, w1):
    ax[2].axvspan(e["start"], e["end"], facecolor="none", edgecolor="k",
                  hatch="xx", lw=0)
ax[2].set(ylabel="fraction of normal", xlabel="time (s)", ylim=(0, 1.45),
          title="Airflow drop vs thresholds   (hatched = DETECTED events)")
ax[2].legend(loc="upper right", fontsize=8)

plt.tight_layout()
plt.show()

## 🔬 Bonus: the real front-end signal (from a thermal clip)

The apnea detection above ran on **synthetic** signals on purpose: real,
labelled overnight apnea recordings are clinical and access-restricted
(ApneaEye was evaluated at AIIMS New Delhi). What we *can* show with public lab
data is the **front-end** — recovering the nasal-airflow breathing waveform from
a **real thermal recording** (ApneaEye demo,
[github.com/AyushShrivstava/ApneaEye_Demo](https://github.com/AyushShrivstava/ApneaEye_Demo)).
The apnea / hypopnea logic above operates on exactly this kind of signal; in the
real system a second signal (thorax–abdomen motion) is recovered the same way.

In [ ]:
import urllib.request, os
try:
    import cv2
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "opencv-python-headless"], check=True)
    import cv2
from scipy.signal import detrend, butter, filtfilt, find_peaks

THERMAL_URL = "https://raw.githubusercontent.com/AyushShrivstava/ApneaEye_Demo/main/Data/demo_ApneaEye_2025-11-12_01-18-18.avi"
thermal_path = "/tmp/demo_apnea_thermal.avi"
have_thermal = False
try:
    if not os.path.exists(thermal_path) or os.path.getsize(thermal_path) < 10000:
        print("Downloading real thermal clip (~15 MB) ...")
        urllib.request.urlretrieve(THERMAL_URL, thermal_path)
    have_thermal = os.path.getsize(thermal_path) > 10000
except Exception as e:
    print("Could not fetch real data (offline?). Skipping.", e)

if have_thermal:
    cap = cv2.VideoCapture(thermal_path)
    fps_r = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(fr.mean(axis=2))
    cap.release()
    vid = np.asarray(frames); T, Hh, Ww = vid.shape

    def band_info(sig, fs, lo=0.1, hi=0.6):
        x = detrend(sig); N = len(x)
        f = np.fft.rfftfreq(N, 1.0 / fs); P = np.abs(np.fft.rfft(x * np.hanning(N))) ** 2
        b = (f >= lo) & (f <= hi)
        return P[b].sum() / P.sum(), f[b][np.argmax(P[b])]

    best = None; rh, rw = Hh // 8, Ww // 10
    for fy in np.linspace(0.25, 0.75, 6):
        for fx in np.linspace(0.30, 0.70, 6):
            cy, cx = int(Hh * fy), int(Ww * fx)
            patch = vid[:, max(0, cy - rh):cy + rh, max(0, cx - rw):cx + rw].mean(axis=(1, 2))
            frac, fpk = band_info(patch, fps_r)
            if best is None or frac > best[0]:
                best = (frac, patch, fpk)
    frac, airflow, fpk = best
    ny = fps_r / 2.0
    bb, aa = butter(3, [0.1 / ny, min(0.6, 0.99 * ny) / ny], btype="band")
    airflow_f = filtfilt(bb, aa, detrend(airflow))

    print("Real nasal-airflow signal recovered from thermal video")
    print("  duration                 : %.0f s @ %.0f fps" % (T / fps_r, fps_r))
    print("  breathing-band power frac : %.2f" % frac)
    print("  breathing rate            : %.1f breaths/min" % (fpk * 60))

    tt = np.arange(len(airflow_f)) / fps_r
    fig, ax = plt.subplots(2, 1, figsize=(11, 5))
    ax[0].plot(tt, airflow_f, color="#1f77b4", lw=1)
    ax[0].set_title("Real nasal-airflow waveform (thermal, ApneaEye demo)")
    ax[0].set_xlabel("time (s)"); ax[0].set_ylabel("a.u.")
    x = detrend(airflow); N = len(x)
    f = np.fft.rfftfreq(N, 1.0 / fps_r); P = np.abs(np.fft.rfft(x * np.hanning(N))) ** 2
    m = (f * 60 >= 5) & (f * 60 <= 40)
    ax[1].plot(f[m] * 60, P[m] / P[m].max(), color="#2ca02c")
    ax[1].axvline(fpk * 60, color="r", ls="--", label="%.1f brpm" % (fpk * 60))
    ax[1].set_title("Spectrum"); ax[1].set_xlabel("breaths/min"); ax[1].legend()
    plt.tight_layout(); plt.show()
    print("\nDuring an apnea this waveform would flatten (>=80% amplitude drop) "
          "for >=10 s — the event rule applied to the synthetic night above.")


## 9. Limitations, exercises, and the safety line

**What this toy captures well**
- The *logic* of two-signal scoring: airflow says *whether* an event happened; thorax effort says *which kind*.
- Why a robust **baseline** matters — you score *relative* drops, not absolute amplitude.
- How event counts roll up into the **AHI** and a severity grade, and why a conservative (over-estimating) detector is the safer failure mode in screening.

**What real ApneaEye data adds (and this notebook skips)**
- Real thermal video is noisy, low-resolution, and the person *moves*; extracting clean airflow/effort signals (temperature ROI tracking, optical flow) is the hard part we assumed away.
- Real sleep also has arousals, position changes, mouth breathing, and signal dropouts.
- Clinical scoring additionally uses oxygen desaturation and arousals to define hypopneas.

**Exercises**
1. **Sensitivity to threshold.** Sweep the apnea threshold from 0.10 to 0.30 and the hypopnea threshold from 0.50 to 0.80. Plot precision and sensitivity vs threshold. Where is the best trade-off, and which direction is "safer" for screening?
2. **Effort matters.** In `make_overnight`, change one obstructive apnea's thorax factor so effort *also* fades. Re-run: does the detector now mislabel it as central? This is exactly the failure you get if your camera loses the thorax signal.
3. **Noisier nights.** Raise `noise` (e.g. 0.04 → 0.12) and add more decoys. Watch precision fall and the AHI drift upward. What post-processing (e.g. a minimum gap between events, or requiring 12 s instead of 10 s) recovers precision?
4. **Min-duration rule.** Inject several 6–9 s dips. Confirm the ≥ 10 s rule rejects them, then break the rule (set `min_dur_s=4`) and watch the false AHI explode.

---
> ⚠️ **Safety.** This notebook is an educational illustration of the ApneaEye *technique* on synthetic data. It is **not** a medical device, it is **not** validated, and it must **not** be used for diagnosis or any health decision. If you are worried about sleep apnea, see a clinician.

*Made for the ACM Summer School on AI for Health.*